# Format and save Supplementary Data Tables from raw data files.

Generates all supplementary data tables (see [results](results/)) except Supplementary Table 8, which is generated in [`Upset Plots.ipynb`](scripts/07_figures_tables/UpSet%20Plots.ipynb).

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import norm
!{sys.executable} -m pip install openpyxl

# Resolve repo root whether cwd is repo root, scripts/, or a stage subdirectory.
cwd = os.getcwd()
if os.path.basename(cwd) == "orb-selection":
    repo_root = cwd
elif os.path.basename(os.path.dirname(cwd)) == "orb-selection":
    repo_root = os.path.dirname(cwd)
elif os.path.basename(os.path.dirname(os.path.dirname(cwd))) == "orb-selection":
    repo_root = os.path.dirname(os.path.dirname(cwd))
else:
    repo_root = cwd

data = os.path.join(repo_root, "data")
results = os.path.join(repo_root, "results")

src_path = os.path.join(repo_root, "src")
stage03_path = os.path.join(repo_root, "scripts", "03_selection_tests")
stage04_path = os.path.join(repo_root, "scripts", "04_permulation_loss_dup")
for path in (src_path, stage03_path, stage04_path):
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Using src path: {src_path}")
print(f"Using stage-03 path: {stage03_path}")
print(f"Using stage-04 path: {stage04_path}")

# Import the modules from stage-03/
from hyphy_results_parser import (
    RelaxResult,
    BustedPhResult
)
from id_converter import convert_hogs_to_locs, get_ptep_description

pd.set_option('display.max_columns', None, 'display.max_rows', 20)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Using src path: /Users/calvin/orb-selection/src
Using stage-03 path: /Users/calvin/orb-selection/scripts/03_selection_tests
Using stage-04 path: /Users/calvin/orb-selection/scripts/04_permulation_loss_dup


/Users/calvin/anaconda3/envs/orb-selection/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext autoreload

In [3]:
def move_column(df, col_name, new_pos):
    cols = list(df.columns)
    cols.insert(new_pos, cols.pop(cols.index(col_name)))
    return df[cols]

## Hyphy Results

In [21]:
hyphy_results = os.path.join(repo_root, "results/hyphy_results_cache/")

# Load the saved RELAX results
relax_result = RelaxResult.load_from_pickle(str(hyphy_results + "relax_results.pkl"))
relax_result_fltrd = relax_result.filter_omega(10000)
relax_hits_df = relax_result_fltrd.get_significant_results()

In [22]:
relax_hits_df = move_column(relax_hits_df, 'result', 0)  # Move 'result' column to the front
relax_hits_df['foreground'] = 'non-orbweavers'
relax_hits_df = move_column(relax_hits_df, 'foreground', 1)  # Move 'foreground' column to the front
relax_hits_df

,result,foreground,p_value,k,LRT,MG94xREV_ω_reference,MG94xREV_ω_test,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,
N5.HOG0067228,relaxed,non-orbweavers,5.069278e-13,0.031330,52.178060,2.038968e-05,1.110457e+00,0.000000,0.843804,0.389555,0.063292,1.000000,0.092904,0.000000,0.843804,8.541217e-14,0.063292,1.000000,0.092904,0.117560,0.092904
N5.HOG0059719,intensified,non-orbweavers,7.772996e-06,1.108589,19.992907,7.502321e-02,6.447394e-02,0.037770,0.404703,0.046494,0.595297,1.964199,0.000000,0.052061,0.404703,6.279584e-02,0.595297,1.838516,0.000000,0.042963,0.058452
N5.HOG0063256,intensified,non-orbweavers,7.733957e-04,1.136232,11.304017,6.619584e-02,5.929911e-02,0.000000,0.810437,0.086857,0.189563,1.002673,0.000000,0.000000,0.810437,1.164229e-01,0.189563,1.002352,0.000000,0.016465,0.022069
N5.HOG0053197,intensified,non-orbweavers,1.414873e-05,1.212468,18.848973,1.444946e-07,1.613655e-07,0.049339,0.277338,0.065543,0.722662,1.003152,0.000000,0.083597,0.277338,1.056603e-01,0.722662,1.002599,0.000000,0.061049,0.099541
N5.HOG0038544,intensified,non-orbweavers,7.216346e-05,1.097131,15.753260,2.865946e+00,4.260989e-06,0.053668,0.706817,0.054859,0.293183,1.002547,0.000000,0.069531,0.706817,7.093601e-02,0.293183,1.002321,0.000000,0.054018,0.069943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067687,intensified,non-orbweavers,9.106801e-04,1.152606,11.000893,1.822100e-01,1.666336e-01,0.099776,0.262095,0.107294,0.737905,2.158247,0.000000,0.135381,0.262095,1.441878e-01,0.737905,1.949243,0.000000,0.105324,0.141880
N5.HOG0064241,intensified,non-orbweavers,2.168763e-02,1.117584,5.270673,5.236016e-05,4.930010e-01,0.000000,0.302399,0.152434,0.697601,1.862849,0.000000,0.000000,0.302399,1.857939e-01,0.697601,1.744823,0.000000,0.106338,0.129610
N5.HOG0066936,intensified,non-orbweavers,6.469430e-05,1.122134,15.960024,1.096739e-07,8.254460e-01,0.012756,0.285540,0.088955,0.714460,1.001747,0.000000,0.020506,0.285540,1.157564e-01,0.714460,1.001557,0.000000,0.067197,0.088559


In [23]:
# Load the saved BUSTED-PH results
busted_ph_result = BustedPhResult.load_from_pickle(str(hyphy_results + "busted_ph_results.pkl"))
busted_ph_result_fltrd = busted_ph_result.filter_omega(10000)
busted_ph_hits_df = busted_ph_result_fltrd.get_significant_results()
busted_ph_hits_df['foreground'] = 'orbweavers'
busted_ph_hits_df = move_column(busted_ph_hits_df, 'foreground', 0)  # Move 'foreground' column to the front
busted_ph_hits_df.drop(columns=['result'], inplace=True)  # Drop result column
busted_ph_hits_df

,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476
N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075
N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874
N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035
N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0059005,orbweavers,0.000000e+00,205.496307,0.500000,0.000000,1.873424e-07,39.511472,0.316642,2.354017e-01,0.001939,0.451225,0.473936,0.502813,19.647255,0.045962,0.000000,0.602167,0.000579,0.000000,0.555289,0.397833,1.142205,0.220912
N5.HOG0056667,orbweavers,3.393248e-08,33.011492,0.500000,0.000000,5.346662e-06,32.231194,0.157910,1.543083e-01,0.131782,0.991548,0.133366,0.006880,160.277082,0.001573,0.007861,0.328270,0.083886,0.489142,0.452285,0.182587,0.383626,0.126194
N5.HOG0068073,orbweavers,4.157678e-04,14.184473,0.500000,0.000000,1.621171e-03,19.396446,0.231503,2.064418e-01,0.026364,0.018939,0.210999,0.978294,455.127141,0.002767,0.099528,0.855957,0.218815,0.036574,1.000000,0.107469,1.466394,0.200663


In [24]:
# Load the saved BUSTED-PH-rev results
busted_ph_rev_result = BustedPhResult.load_from_pickle(str(hyphy_results + "busted_ph_rev_results.pkl"))
busted_ph_rev_result_fltrd = busted_ph_rev_result.filter_omega(10000)
busted_ph_rev_hits_df = busted_ph_rev_result_fltrd.get_significant_results()
busted_ph_rev_hits_df['foreground'] = 'non-orbweavers'
busted_ph_rev_hits_df = move_column(busted_ph_rev_hits_df, 'foreground', 0)  # Move 'foreground' column to the front
busted_ph_rev_hits_df.drop(columns=['result'], inplace=True)  # Drop result column
busted_ph_rev_hits_df

,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0019712,non-orbweavers,0.000000e+00,809.032492,0.500000,0.000000,0.000000e+00,449.467708,0.000040,0.000017,0.028944,0.645432,0.312607,0.330121,182.484806,0.024447,0.000000,0.071261,0.014606,0.826985,1.000000,0.101753,4.583093,0.113832
N5.HOG0056026,non-orbweavers,8.155119e-10,40.468116,0.500000,0.000000,5.973391e-04,21.698708,0.135780,0.128210,0.051367,0.857685,0.500453,0.139372,5258.822922,0.002943,0.008322,0.719924,0.185418,0.194951,0.859716,0.085125,15.589621,0.115322
N5.HOG0062863,non-orbweavers,3.469447e-14,60.596936,0.500000,0.000000,3.956550e-04,22.638929,0.211636,0.162953,0.081360,0.987843,0.738062,0.007109,4553.459205,0.005048,0.000000,0.068601,0.038564,0.818016,0.530455,0.113383,23.070581,0.091690
N5.HOG0041381,non-orbweavers,0.000000e+00,102.987142,0.109140,3.043957,6.191836e-06,31.909387,0.166545,0.161066,0.000000,0.006265,0.133910,0.987070,25007.129413,0.006665,0.000000,0.381828,0.155732,0.584634,1.691232,0.033538,166.802487,0.147767
N5.HOG0036458,non-orbweavers,2.299926e-02,6.158292,0.133744,2.637359,3.273738e-04,23.069636,0.005095,0.007983,0.001590,0.904070,0.003990,0.095343,4872.117587,0.000587,0.003572,0.994320,0.003753,0.003940,2.572762,0.001740,2.860019,0.008043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,0.000035,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647
N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,0.000077,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921
N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,0.207552,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395


In [25]:
all_busted_hits_df = pd.concat([busted_ph_hits_df, busted_ph_rev_hits_df])
all_busted_hits_df

,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref
HOG,,,,,,,,,,,,,,,,,,,,,,,
N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476
N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075
N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874
N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035
N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647
N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921
N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395


In [26]:
busted_hits_locs = convert_hogs_to_locs(all_busted_hits_df, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
busted_hits_locs_fltrd = busted_hits_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
busted_hits_locs_fltrd = move_column(busted_hits_locs_fltrd, 'HOG', 0)
busted_hits_locs_fltrd.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
busted_hits_locs_fltrd

Processing HOGs: 100%|██████████| 292/292 [00:00<00:00, 25150.66it/s]


,HOG,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,U. diversus gene ID,U. diversus GO terms,U. diversus gene description
0,N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476,129215997,GO:0006470 [Name: protein dephosphorylation]; ...,TGF-beta-activated kinase 1 and MAP3K7-binding...
1,N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075,129220007,GO:0045944 [Name: positive regulation of trans...,RING finger protein 10-like
2,N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874,129222444,GO:0001405 [Name: presequence translocase-asso...,"grpE protein homolog, mitochondrial Roe1"
3,N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035,129218644,GO:0000407 [Name: pre-autophagosomal structure...,autophagy-related protein 13-like
4,N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945,129231803,GO:0005515 [Name: protein binding],leucine-rich repeat-containing protein 59-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647,129225838,GO:0042273 [Name: ribosomal large subunit biog...,nucleolar complex protein 2 homolog
288,N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921,NaN,NaN,NaN
289,N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395,129223042,GO:0000981 [Name: RNA polymerase II transcript...,uncharacterized LOC129223042
290,N5.HOG0063651,non-orbweavers,2.807126e-05,19.575234,0.500000,0.000000,1.631340e-05,29.777420,0.000025,3.369798e-05,0.022216,0.824099,0.370172,0.170712,21.378347,0.005190,0.000000,0.540813,0.212053,0.376480,0.853517,0.082707,0.192446,0.150425,129220117,-,uncharacterized LOC129220117


In [27]:
busted_df = get_ptep_description(busted_hits_locs_fltrd)
busted_df

,HOG,foreground,test_pval,test_LRT,background_pval,background_LRT,shared_pval,shared_LRT,MG94xREV_ω_test,MG94xREV_ω_ref,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0058262,orbweavers,1.545704e-03,11.558257,0.194303,1.890383,2.477354e-02,12.855262,0.194701,1.595677e-01,0.000000,0.561941,0.187650,0.437508,41.105141,0.000551,0.000000,0.425298,0.108554,0.561794,1.974842,0.012908,0.104760,0.086476,129215997,GO:0006470 [Name: protein dephosphorylation]; ...,TGF-beta-activated kinase 1 and MAP3K7-binding...,TGF-beta-activated kinase 1 and MAP3K7-binding...
1,N5.HOG0059024,orbweavers,1.246802e-03,11.988053,0.500000,0.000000,1.286804e-02,14.473037,0.473107,1.181737e-07,0.000000,0.270546,0.185910,0.702125,3.052891,0.027329,0.064841,0.000000,0.065182,0.898490,1.000000,0.101510,0.213964,0.160075,129220007,GO:0045944 [Name: positive regulation of trans...,RING finger protein 10-like,E3 ubiquitin-protein ligase RNF10 isoform X1
2,N5.HOG0062008,orbweavers,4.821284e-08,32.308987,0.408194,0.405731,2.667662e-07,38.749055,0.308877,2.457620e-01,0.000000,0.097881,0.159755,0.897586,48.943871,0.004533,0.062739,0.753940,0.082826,0.189299,1.319449,0.056761,0.365235,0.137874,129222444,GO:0001405 [Name: presequence translocase-asso...,"grpE protein homolog, mitochondrial Roe1","grpE protein homolog, mitochondrial"
3,N5.HOG0032852,orbweavers,1.303768e-11,48.740062,0.477842,0.090656,9.747535e-04,20.573963,0.154262,1.945747e-01,0.009604,0.626402,0.217712,0.372822,3967.329535,0.000775,0.001545,0.069786,0.080352,0.886788,1.120800,0.043426,3.162785,0.120035,129218644,GO:0000407 [Name: pre-autophagosomal structure...,autophagy-related protein 13-like,autophagy-related protein 13 homolog
4,N5.HOG0060473,orbweavers,7.599450e-08,31.398915,0.500000,0.000000,4.055495e-04,22.582696,0.239854,2.098764e-01,0.044892,0.586331,0.490225,0.412124,510139.703369,0.001545,0.001943,0.318434,0.109385,0.427393,0.623890,0.254173,788.217417,0.205945,129231803,GO:0005515 [Name: protein binding],leucine-rich repeat-containing protein 59-like,leucine-rich repeat-containing protein 59 isof...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,N5.HOG0067033,non-orbweavers,0.000000e+00,104.322215,0.500000,-20.557023,0.000000e+00,111.260119,0.000032,3.549836e-05,0.000000,0.461302,0.166979,0.534538,178169.075985,0.004160,0.000000,0.485872,0.158264,0.494977,1.948169,0.019151,741.286697,0.115647,129225838,GO:0042273 [Name: ribosomal large subunit biog...,nucleolar complex protein 2 homolog,nucleolar complex protein 2 homolog
288,N5.HOG0065829,non-orbweavers,0.000000e+00,197.638764,0.500000,-844.964242,2.065015e-11,58.885575,0.000105,7.708236e-05,0.003391,0.582894,0.483803,0.369519,13.393360,0.047587,0.000000,0.282822,0.045675,0.507407,1.848422,0.209771,0.818094,0.410921,NaN,NaN,NaN,uncharacterized protein
289,N5.HOG0067927,non-orbweavers,0.000000e+00,146.664449,0.500000,0.000000,4.634104e-11,57.183980,0.192987,2.075518e-01,0.035514,0.893746,0.701318,0.089573,6248.594546,0.016681,0.000000,0.030754,0.031915,0.809310,0.728830,0.159936,104.326700,0.142395,129223042,GO:0000981 [Name: RNA polymerase II transcript...,uncharacterized LOC129223042,T-cell acute lymphocytic leukemia protein 1 ho...
290,N5.HOG0063651,non-orbweavers,2.807126e-05,19.575234,0.500000,0.000000,1.631340e-05,29.777420,0.000025,3.369798e-05,0.022216,0.824099,0.370172,0.170712,21.378347,0.005190,0.000000,0.540813,0.212053,0.376480,0.853517,0.082707,0.192446,0.150425,129220117,-,uncharacterized LOC129220117,uncharacterized protein


In [28]:
relax_hits_locs = convert_hogs_to_locs(relax_hits_df, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
relax_hits_locs_fltrd = relax_hits_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
relax_hits_locs_fltrd = move_column(relax_hits_locs_fltrd, 'HOG', 0)
relax_hits_locs_fltrd.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
relax_df = get_ptep_description(relax_hits_locs_fltrd)
relax_df

Processing HOGs: 100%|██████████| 1429/1429 [00:00<00:00, 55979.94it/s]


,HOG,result,foreground,p_value,k,LRT,MG94xREV_ω_reference,MG94xREV_ω_test,ω1_test,ω1_test_P,ω2_test,ω2_test_P,ω3_test,ω3_test_P,ω1_ref,ω1_ref_P,ω2_ref,ω2_ref_P,ω3_ref,ω3_ref_P,ω_mean_test,ω_mean_ref,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0067228,relaxed,non-orbweavers,5.069278e-13,0.031330,52.178060,2.038968e-05,1.110457e+00,0.000000,0.843804,0.389555,0.063292,1.000000,0.092904,0.000000,0.843804,8.541217e-14,0.063292,1.000000,0.092904,0.117560,0.092904,129223414,GO:0005737 [Name: cytoplasm]; GO:0016925 [Name...,activator of SUMO 1,SUMO-activating enzyme subunit 1
1,N5.HOG0059719,intensified,non-orbweavers,7.772996e-06,1.108589,19.992907,7.502321e-02,6.447394e-02,0.037770,0.404703,0.046494,0.595297,1.964199,0.000000,0.052061,0.404703,6.279584e-02,0.595297,1.838516,0.000000,0.042963,0.058452,129227349,GO:0005737 [Name: cytoplasm]; GO:0042803 [Name...,serine hydroxymethyltransferase-like,serine hydroxymethyltransferase
2,N5.HOG0063256,intensified,non-orbweavers,7.733957e-04,1.136232,11.304017,6.619584e-02,5.929911e-02,0.000000,0.810437,0.086857,0.189563,1.002673,0.000000,0.000000,0.810437,1.164229e-01,0.189563,1.002352,0.000000,0.016465,0.022069,129224590,GO:0003924 [Name: GTPase activity]; GO:0005737...,GPN-loop GTPase 2-like,GPN-loop GTPase 2 isoform X2
3,N5.HOG0053197,intensified,non-orbweavers,1.414873e-05,1.212468,18.848973,1.444946e-07,1.613655e-07,0.049339,0.277338,0.065543,0.722662,1.003152,0.000000,0.083597,0.277338,1.056603e-01,0.722662,1.002599,0.000000,0.061049,0.099541,129220170,GO:0005634 [Name: nucleus]; GO:0032502 [Name: ...,molecular chaperone MKKS-like,chromatin accessibility complex protein 1
4,N5.HOG0038544,intensified,non-orbweavers,7.216346e-05,1.097131,15.753260,2.865946e+00,4.260989e-06,0.053668,0.706817,0.054859,0.293183,1.002547,0.000000,0.069531,0.706817,7.093601e-02,0.293183,1.002321,0.000000,0.054018,0.069943,NaN,NaN,NaN,frizzled-4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1424,N5.HOG0067687,intensified,non-orbweavers,9.106801e-04,1.152606,11.000893,1.822100e-01,1.666336e-01,0.099776,0.262095,0.107294,0.737905,2.158247,0.000000,0.135381,0.262095,1.441878e-01,0.737905,1.949243,0.000000,0.105324,0.141880,NaN,NaN,NaN,trithorax group protein osa
1425,N5.HOG0064241,intensified,non-orbweavers,2.168763e-02,1.117584,5.270673,5.236016e-05,4.930010e-01,0.000000,0.302399,0.152434,0.697601,1.862849,0.000000,0.000000,0.302399,1.857939e-01,0.697601,1.744823,0.000000,0.106338,0.129610,129227186,"GO:0006352 [Name: DNA-templated transcription,...",TATA box-binding protein-like 1,TATA box-binding protein-like 1
1426,N5.HOG0066936,intensified,non-orbweavers,6.469430e-05,1.122134,15.960024,1.096739e-07,8.254460e-01,0.012756,0.285540,0.088955,0.714460,1.001747,0.000000,0.020506,0.285540,1.157564e-01,0.714460,1.001557,0.000000,0.067197,0.088559,129234669,GO:0009058 [Name: biosynthetic process]; GO:00...,eukaryotic translation initiation factor 2B su...,translation initiation factor eIF2B subunit gamma
1427,N5.HOG0060161,intensified,non-orbweavers,1.160829e-03,1.145080,10.551673,5.080422e-02,6.864197e-02,0.000000,0.876649,1.000000,0.118220,2318.115008,0.005132,0.000000,0.876649,1.000000e+00,0.118220,868.514944,0.005132,12.013916,4.575112,129226064,GO:0005515 [Name: protein binding]; GO:0007098...,partitioning defective protein 6,partitioning defective 6 homolog gamma


In [29]:
relax_df.to_excel(os.path.join(repo_root, "results/Supplementary_Table_3_RELAX_hits.xlsx"), index=False)
relax_df.to_excel(os.path.join(repo_root, "results/Supplementary_Table_4_BUSTED_hits.xlsx"), index=False)

## Permutation test results

before

In [8]:
%autoreload 2
perm_test_results_dir = os.path.join(repo_root, "results/odds_ratio_test/Results_Apr16/Run1_occ_30-88_10000x_all_orb/")
perm_results_old = pd.read_csv(os.path.join(perm_test_results_dir, "fltrd_hits.csv"), index_col="HOG")

perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("loss_fg"), "Loss more likely in orb-weavers", None)
perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("loss_bg"), "Loss more likely in non-orb-weavers", perm_results_old["Result"])
perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("dup_fg"), "Duplication more likely in orb-weavers", perm_results_old["Result"])
perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("dup_bg"), "Duplication more likely in non-orb-weavers", perm_results_old["Result"])
perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_bg"), "Loss more likely in orb-weavers, duplication more likely in non-orb-weavers", perm_results_old["Result"])
perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_fg"), "Loss more likely in non-orb-weavers, duplication more likely in orb-weavers", perm_results_old["Result"])
perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_fg"), "Loss more likely in orb-weavers, duplication more likely in orb-weavers", perm_results_old["Result"])
perm_results_old["Result"] = np.where(perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_bg"), "Loss more likely in non-orb-weavers, duplication more likely in non-orb-weavers", perm_results_old["Result"])

perm_results_old = move_column(perm_results_old, "Result", 0)  # Move 'Result' column to the front
perm_results_old["LOR outside permuted avg thresholds"] = np.where(perm_results_old["Significant by avgd thresholds"].notnull(), "Yes", "No")
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_fg")), "Yes for loss; no for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results_old["Significant by avgd thresholds"].str.contains("dup_bg")), "No for loss; yes for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_fg, dup_bg")), "Yes for both tests", perm_results_old["LOR outside permuted avg thresholds"])

perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_bg")), "Yes for loss; no for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results_old["Significant by avgd thresholds"].str.contains("dup_fg")), "No for loss; yes for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_bg, dup_fg")), "Yes for both tests", perm_results_old["LOR outside permuted avg thresholds"])


perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_fg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_fg")), "Yes for loss; no for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_fg") & perm_results_old["Significant by avgd thresholds"].str.contains("dup_fg")), "No for loss; yes for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_fg, dup_fg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_fg, dup_fg")), "Yes for both tests", perm_results_old["LOR outside permuted avg thresholds"])


perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_bg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_bg")), "Yes for loss; no for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_bg") & perm_results_old["Significant by avgd thresholds"].str.contains("dup_bg")), "No for loss; yes for duplication", perm_results_old["LOR outside permuted avg thresholds"])
perm_results_old["LOR outside permuted avg thresholds"] = np.where((perm_results_old["Significant by permulation"].str.contains("loss_bg, dup_bg") & perm_results_old["Significant by avgd thresholds"].str.contains("loss_bg, dup_bg")), "Yes for both tests", perm_results_old["LOR outside permuted avg thresholds"])


perm_results_old = move_column(perm_results_old, "LOR outside permuted avg thresholds", 4)  # Move 'LOR outside permuted avg thresholds' column to the front
perm_results_old = perm_results_old.drop(columns=["Significant by avgd thresholds", "Significant by permulation"])
perm_results_old = perm_results_old.rename(columns={
    "P-value loss more likely in fg": "P-value, loss more likely in orb-weavers", 
    "P-value loss more likely in bg": "P-value, loss more likely in non-orb-weavers", 
    "P-value duplication more likely in fg": "P-value, duplication more likely in orb-weavers", 
    "P-value duplication more likely in bg": "P-value, duplication more likely in non-orb-weavers", 
    })
perm_results_old

,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers","P-value, duplication more likely in orb-weavers","P-value, duplication more likely in non-orb-weavers"
HOG,,,,,,,,,
N5.HOG0001041,Loss more likely in orb-weavers,46,2.288042,-0.281735,No,0.0038,0.9962,0.5328,0.4672
N5.HOG0001609,Loss more likely in non-orb-weavers,49,-2.841587,2.391485,No,0.9605,0.0395,0.0747,0.9253
N5.HOG0001627,"Loss more likely in orb-weavers, duplication m...",33,2.048043,-4.020457,No for loss; yes for duplication,0.0449,0.9551,0.9745,0.0253
N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.823601,1.854638,Yes,0.9996,0.0004,0.3311,0.6686
N5.HOG0002072,Loss more likely in orb-weavers,52,3.473733,-0.762726,Yes,0.0108,0.9892,0.7989,0.2011
...,...,...,...,...,...,...,...,...,...
N5.HOG0073114,Loss more likely in orb-weavers,36,1.310443,-1.206534,No,0.0218,0.9782,0.7961,0.2038
N5.HOG0073157,Loss more likely in non-orb-weavers,34,-3.186391,1.988284,Yes,0.9576,0.0424,0.1151,0.8846
N5.HOG0073180,Loss more likely in non-orb-weavers,32,-1.038058,0.724732,No,0.9929,0.0071,0.1394,0.8606


Reformat results table

In [10]:
%autoreload 2
perm_test_results_dir = os.path.join(repo_root, "results/odds_ratio_test/Results_Jul22/Run1_occ_30-88_10000x/")
perm_results = pd.read_csv(os.path.join(perm_test_results_dir, "fltrd_hits.csv"), index_col="HOG")

perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_fg"), "Loss more likely in orb-weavers", None)
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_bg"), "Loss more likely in non-orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("dup_fg"), "Duplication more likely in orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("dup_bg"), "Duplication more likely in non-orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg"), "Loss more likely in orb-weavers, duplication more likely in non-orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg"), "Loss more likely in non-orb-weavers, duplication more likely in orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_fg, dup_fg"), "Loss more likely in orb-weavers, duplication more likely in orb-weavers", perm_results["Result"])
perm_results["Result"] = np.where(perm_results["Significant by permulation"].str.contains("loss_bg, dup_bg"), "Loss more likely in non-orb-weavers, duplication more likely in non-orb-weavers", perm_results["Result"])

perm_results = move_column(perm_results, "Result", 0)  # Move 'Result' column to the front
perm_results["LOR outside permuted avg thresholds"] = np.where(perm_results["Significant by avgd thresholds"].notnull(), "Yes", "No")
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("loss_fg")), "Yes for loss; no for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("dup_bg")), "No for loss; yes for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("loss_fg, dup_bg")), "Yes for both tests", perm_results["LOR outside permuted avg thresholds"])

perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("loss_bg")), "Yes for loss; no for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("dup_fg")), "No for loss; yes for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("loss_bg, dup_fg")), "Yes for both tests", perm_results["LOR outside permuted avg thresholds"])


perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("loss_fg")), "Yes for loss; no for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("dup_fg")), "No for loss; yes for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_fg, dup_fg") & perm_results["Significant by avgd thresholds"].str.contains("loss_fg, dup_fg")), "Yes for both tests", perm_results["LOR outside permuted avg thresholds"])


perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("loss_bg")), "Yes for loss; no for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("dup_bg")), "No for loss; yes for duplication", perm_results["LOR outside permuted avg thresholds"])
perm_results["LOR outside permuted avg thresholds"] = np.where((perm_results["Significant by permulation"].str.contains("loss_bg, dup_bg") & perm_results["Significant by avgd thresholds"].str.contains("loss_bg, dup_bg")), "Yes for both tests", perm_results["LOR outside permuted avg thresholds"])


perm_results = move_column(perm_results, "LOR outside permuted avg thresholds", 4)  # Move 'LOR outside permuted avg thresholds' column to the front
perm_results = perm_results.drop(columns=["Significant by avgd thresholds", "Significant by permulation"])
perm_results = perm_results.rename(columns={
    "P-value loss more likely in fg": "P-value, loss more likely in orb-weavers", 
    "P-value loss more likely in bg": "P-value, loss more likely in non-orb-weavers", 
    "P-value duplication more likely in fg": "P-value, duplication more likely in orb-weavers", 
    "P-value duplication more likely in bg": "P-value, duplication more likely in non-orb-weavers", 
    })
perm_results

,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers","P-value, duplication more likely in orb-weavers","P-value, duplication more likely in non-orb-weavers"
HOG,,,,,,,,,
N5.HOG0001041,Loss more likely in orb-weavers,46,2.521478,-0.155768,No,0.0009,0.9991,0.4798,0.5202
N5.HOG0001627,Duplication more likely in non-orb-weavers,33,1.855831,-3.857911,Yes,0.0640,0.9360,0.9873,0.0127
N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.395453,1.332164,Yes,0.9984,0.0016,0.4120,0.5880
N5.HOG0002072,Loss more likely in orb-weavers,52,3.079265,-0.639315,Yes,0.0143,0.9857,0.8034,0.1964
N5.HOG0002076,Loss more likely in non-orb-weavers,55,-2.050543,0.725386,No,0.9545,0.0455,0.1584,0.8416
...,...,...,...,...,...,...,...,...,...
N5.HOG0073071,Duplication more likely in orb-weavers,35,-2.150469,2.393444,Yes,0.7460,0.2540,0.0113,0.9887
N5.HOG0073090,Loss more likely in orb-weavers,32,1.527411,-1.712408,No,0.0256,0.9744,0.9150,0.0850
N5.HOG0073180,Loss more likely in non-orb-weavers,32,-1.042447,0.851411,No,0.9953,0.0047,0.1060,0.8940


In [17]:
# intersection of indices
common_idx = perm_results.index.intersection(perm_results_old.index)
intersection_df = perm_results.loc[common_idx]

# quick summary
print(f"Common HOGs: {len(common_idx)}")
intersection_df.head()

Common HOGs: 999


,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers","P-value, duplication more likely in orb-weavers","P-value, duplication more likely in non-orb-weavers"
HOG,,,,,,,,,
N5.HOG0001041,Loss more likely in orb-weavers,46,2.521478,-0.155768,No,0.0009,0.9991,0.4798,0.5202
N5.HOG0001627,Duplication more likely in non-orb-weavers,33,1.855831,-3.857911,Yes,0.0640,0.9360,0.9873,0.0127
N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.395453,1.332164,Yes,0.9984,0.0016,0.4120,0.5880
N5.HOG0002072,Loss more likely in orb-weavers,52,3.079265,-0.639315,Yes,0.0143,0.9857,0.8034,0.1964
N5.HOG0002076,Loss more likely in non-orb-weavers,55,-2.050543,0.725386,No,0.9545,0.0455,0.1584,0.8416


In [12]:
%autoreload 2
perm_locs = convert_hogs_to_locs(perm_results, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
perm_locs_filtered = perm_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
perm_locs_filtered = move_column(perm_locs_filtered, 'HOG', 0)
perm_locs_filtered.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
perm_locs_filtered

Processing HOGs: 100%|██████████| 1482/1482 [00:00<00:00, 61000.57it/s]


,HOG,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers","P-value, duplication more likely in orb-weavers","P-value, duplication more likely in non-orb-weavers",U. diversus gene ID,U. diversus GO terms,U. diversus gene description
0,N5.HOG0001041,Loss more likely in orb-weavers,46,2.521478,-0.155768,No,0.0009,0.9991,0.4798,0.5202,NaN,NaN,NaN
1,N5.HOG0001627,Duplication more likely in non-orb-weavers,33,1.855831,-3.857911,Yes,0.0640,0.9360,0.9873,0.0127,NaN,NaN,NaN
2,N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.395453,1.332164,Yes,0.9984,0.0016,0.4120,0.5880,129233975,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like
3,N5.HOG0002072,Loss more likely in orb-weavers,52,3.079265,-0.639315,Yes,0.0143,0.9857,0.8034,0.1964,129217362,GO:0008010 [Name: structural constituent of ch...,proteoglycan 4-like
4,N5.HOG0002076,Loss more likely in non-orb-weavers,55,-2.050543,0.725386,No,0.9545,0.0455,0.1584,0.8416,129221291,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,N5.HOG0073071,Duplication more likely in orb-weavers,35,-2.150469,2.393444,Yes,0.7460,0.2540,0.0113,0.9887,NaN,NaN,NaN
1478,N5.HOG0073090,Loss more likely in orb-weavers,32,1.527411,-1.712408,No,0.0256,0.9744,0.9150,0.0850,NaN,NaN,NaN
1479,N5.HOG0073180,Loss more likely in non-orb-weavers,32,-1.042447,0.851411,No,0.9953,0.0047,0.1060,0.8940,129221249,-,cuticle protein 38-like
1480,N5.HOG0073332,Loss more likely in non-orb-weavers,32,-3.602439,0.354166,Yes,0.9701,0.0299,0.4704,0.5296,NaN,NaN,NaN


In [13]:
%autoreload 2
perm_df = get_ptep_description(perm_locs_filtered)
perm_df

,HOG,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers","P-value, duplication more likely in orb-weavers","P-value, duplication more likely in non-orb-weavers",U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0001041,Loss more likely in orb-weavers,46,2.521478,-0.155768,No,0.0009,0.9991,0.4798,0.5202,NaN,NaN,NaN,gastrula zinc finger protein XlCGF7.1
1,N5.HOG0001627,Duplication more likely in non-orb-weavers,33,1.855831,-3.857911,Yes,0.0640,0.9360,0.9873,0.0127,NaN,NaN,NaN,speckle-type POZ protein
2,N5.HOG0002031,Loss more likely in non-orb-weavers,46,-3.395453,1.332164,Yes,0.9984,0.0016,0.4120,0.5880,129233975,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like,cuticle protein 10.9
3,N5.HOG0002072,Loss more likely in orb-weavers,52,3.079265,-0.639315,Yes,0.0143,0.9857,0.8034,0.1964,129217362,GO:0008010 [Name: structural constituent of ch...,proteoglycan 4-like,uncharacterized protein
4,N5.HOG0002076,Loss more likely in non-orb-weavers,55,-2.050543,0.725386,No,0.9545,0.0455,0.1584,0.8416,129221291,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like,cuticle protein 10.9-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,N5.HOG0073071,Duplication more likely in orb-weavers,35,-2.150469,2.393444,Yes,0.7460,0.2540,0.0113,0.9887,NaN,NaN,NaN,uncharacterized protein
1478,N5.HOG0073090,Loss more likely in orb-weavers,32,1.527411,-1.712408,No,0.0256,0.9744,0.9150,0.0850,NaN,NaN,NaN,DNA mismatch repair protein Mlh3
1479,N5.HOG0073180,Loss more likely in non-orb-weavers,32,-1.042447,0.851411,No,0.9953,0.0047,0.1060,0.8940,129221249,-,cuticle protein 38-like,cuticle protein 63-like
1480,N5.HOG0073332,Loss more likely in non-orb-weavers,32,-3.602439,0.354166,Yes,0.9701,0.0299,0.4704,0.5296,NaN,NaN,NaN,NO_HIT


In [18]:
perm_df.to_excel(os.path.join(repo_root, 'results/Supplementary_Table_5_OddsRatioTest_hits.xlsx'), index=False)

In [14]:
previous_interesting = pd.read_csv(os.path.join(repo_root, 'data/interesting_hits.txt'), header=None, names=['HOG'])

In [15]:
previous_interesting_df = perm_df[perm_df['HOG'].isin(previous_interesting['HOG'])]
previous_interesting_df.to_excel(os.path.join(repo_root, 'results/Supplementary_Table_6_OddsRatioTest_interesting_hits.xlsx'), index=False)
previous_interesting_df

,HOG,Result,Occupancy,Log odds ratio of loss,Log odds ratio of duplication,LOR outside permuted avg thresholds,"P-value, loss more likely in orb-weavers","P-value, loss more likely in non-orb-weavers","P-value, duplication more likely in orb-weavers","P-value, duplication more likely in non-orb-weavers",U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
43,N5.HOG0007453,Loss more likely in orb-weavers,50,1.946870,-1.544702,No,0.0165,0.9835,0.9383,0.0617,129216356,GO:0043005 [Name: neuron projection],lachesin-like,lachesin
113,N5.HOG0014159,Duplication more likely in orb-weavers,78,0.169513,2.093720,No,0.3622,0.6378,0.0394,0.9606,129217691,GO:0005109 [Name: frizzled binding]; GO:000557...,protein Wnt-16-like,protein Wnt-16-like precursor
175,N5.HOG0018539,Loss more likely in non-orb-weavers,84,-1.480659,0.395499,No,0.9706,0.0294,0.3609,0.6391,129230744,GO:0005737 [Name: cytoplasm]; GO:0097320 [Name...,protein kinase C and casein kinase substrate i...,protein kinase C and casein kinase substrate i...
228,N5.HOG0022330,Loss more likely in non-orb-weavers,50,-2.722522,2.141345,No,0.9938,0.0062,0.0579,0.9421,129217545,GO:0046777 [Name: protein autophosphorylation]...,dual specificity tyrosine-phosphorylation-regu...,dual specificity tyrosine-phosphorylation-regu...
260,N5.HOG0024065,"Loss more likely in non-orb-weavers, duplicati...",66,-2.850324,2.026191,Yes for loss; no for duplication,0.9583,0.0417,0.0115,0.9885,129218972,GO:0005737 [Name: cytoplasm]; GO:0006468 [Name...,uncharacterized LOC129218972,uncharacterized protein
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1124,N5.HOG0063077,Duplication more likely in non-orb-weavers,96,-1.260838,-1.784818,No,0.7608,0.2392,0.9733,0.0267,129219268,GO:0043025 [Name: neuronal cell body]; GO:0045...,fragile X messenger ribonucleoprotein 1 homolog,RNA-binding protein FXR1
1268,N5.HOG0066939,Loss more likely in non-orb-weavers,69,-2.650704,0.939259,No,0.9714,0.0286,0.0699,0.9301,129228009,GO:0015250 [Name: water channel activity]; GO:...,neurogenic protein big brain-like,aquaporin AQPAe.a
1275,N5.HOG0067099,Duplication more likely in non-orb-weavers,56,2.257462,-2.780946,Yes,0.0830,0.9170,0.9689,0.0311,129225034,GO:0005737 [Name: cytoplasm]; GO:0019901 [Name...,Cdk5 activator-like protein,cyclin-dependent kinase 5 activator 1 isoform X1
1347,N5.HOG0068715,Loss more likely in non-orb-weavers,83,-2.162736,0.712293,No,0.9910,0.0090,0.2442,0.7558,129232140,GO:0004930 [Name: G-protein coupled receptor a...,probable G-protein coupled receptor AH9.1,probable G-protein coupled receptor AH9.1


## Spider accessions and BUSCO scores  

In [36]:
buscos = pd.read_csv(f'{data}/orthorun_list_fams_busco90.csv')
accessions = pd.read_csv(f'{data}/txptome_accessions.csv')
accessions.drop(columns=['Infraorder', 'Family', 'Web_Type', 'Cribellum'], inplace=True)

buscos_accessions = pd.merge(buscos, accessions, on='Species', how='left')

buscos_accessions.drop_duplicates(inplace=True)
buscos_accessions.dropna(how='any', inplace=True)
buscos_accessions.reset_index(drop=True, inplace=True)
# Collapse rows: group by all columns except 'Accession', join 'Accession' values as comma-separated string
buscos_accessions.rename(columns={'Single': 'Single_copy_BUSCOs', 'Duplicated': 'Duplicated_BUSCOs', 'Total': 'Total_BUSCOs'}, inplace=True)
buscos_accessions.shape[0]


102

In [37]:
buscos_accessions.to_excel(f'{results}/Supplementary_Table_1_SpiderAccessions_BUSCOs.xlsx', index=False)


In [38]:
# Count rows with NaN in the 'Accession' column before collapsing
num_nan_accession = buscos_accessions['Accession'].isna().sum()
print(f"Rows with NaN in 'Accession': {num_nan_accession}")

Rows with NaN in 'Accession': 0


## N5 HOGs

this is just to make the format match the others

In [39]:
n5_hogs = pd.read_csv(f'{data}/N5.tsv', sep='\t')
n5_hogs.to_excel(f'{results}/Supplementary_Table_2_Entelegyne_HOGs.xlsx', index=False)

/var/folders/h8/wkfyd5dj3nnd_smftqfplclh0000gn/T/ipykernel_19504/1880799938.py:1: DtypeWarning: Columns (0: Acroaspis_sp_IDV6688, 1: Acrosomoides_sp_IDV7426, 2: Ainerigone_saitoi, 3: Ambicodamus_IDV6680, 4: Araneus_inustus, 5: Arcuphantes_paiki, 6: Arcuphantes_tamaensis, 7: Arcuphantes_uenoi, 8: Argiope_aetheroides, 9: Argiope_bruennichi, 10: Argiope_keyserlingi, 11: Argyrodes_flavescens, 12: Argyrodes_miniaceus, 13: Backobourkia_brouni, 14: Baryphymula_kamakuraensis, 15: Caerostris_darwini, 16: Caerostris_extrusa, 17: Chrysso_octomaculata, 18: Clubiona_robusta, 19: Cryptachaea_gigantipes, 20: Cyclosa_octotuberculata, 21: Cyrtarachne_bufo, 22: Cyrtophora_cicatrosa, 23: Deinopis_sp_IDV6783, 24: Deinopis_sp_IDV7365, 25: Desis_marina, 26: Diaea_subdola, 27: Doenitzius_peniculus, 28: Doenitzius_pruvus, 29: Dolophones_sp_IDV6683, 30: Drapetisca_socialis, 31: Ebrechtella_tricuspidata, 32: Eriophora_pustulosa, 33: Floronia_bucculenta, 34: Gnathonarium_exsiccatum, 35: Helpis_minitabunda, 36: H

## PhyloGLM results

In [19]:
phyloglm = pd.read_csv(f'{results}/phyloglm/phyloglm_qvals.csv', index_col="HOG")
phyloglm

,error,coef_.Intercept._Estimate,coef_.Intercept._StdErr,coef_.Intercept._z.value,coef_.Intercept._p.value,coef_orb_weavingTRUE_Estimate,coef_orb_weavingTRUE_StdErr,coef_orb_weavingTRUE_z.value,coef_orb_weavingTRUE_p.value,qvalue
HOG,,,,,,,,,,
N5.HOG0000042,NaN,-1.824872,1.886352,-0.967408,0.333340,-0.501943,1.173294,-0.427807,0.668792,0.611945
N5.HOG0000052,NaN,-0.856693,0.819677,-1.045159,0.295949,0.307564,0.362892,0.847535,0.396697,0.464750
N5.HOG0000147,NaN,-1.239375,0.740910,-1.672775,0.094372,0.030086,0.340894,0.088255,0.929674,0.696301
N5.HOG0000196,NaN,-0.863214,0.494447,-1.745817,0.080843,0.171416,0.220604,0.777031,0.437141,0.491348
N5.HOG0000276,NaN,-2.180273,1.272667,-1.713153,0.086684,0.537627,0.812604,0.661610,0.508221,0.532658
...,...,...,...,...,...,...,...,...,...,...
N5.HOG0073724,NaN,-0.935609,0.563660,-1.659881,0.096938,-0.639715,0.390120,-1.639791,0.101049,0.190854
N5.HOG0073740,NaN,-0.716498,0.376449,-1.903308,0.057000,0.121716,0.169320,0.718848,0.472235,0.513420
N5.HOG0073766,NaN,-1.630470,0.800970,-2.035620,0.041788,1.041255,0.419621,2.481418,0.013086,0.046133


In [20]:
phyloglm_sig = phyloglm[phyloglm['qvalue'] < 0.05].drop(columns=[
    'error', 'coef_.Intercept._Estimate', 'coef_.Intercept._StdErr', 'coef_.Intercept._z.value', 'coef_.Intercept._p.value'])
phyloglm_sig.rename(columns={
    'coef_orb_weavingTRUE_Estimate': 'Orb-weaving coefficient estimate',
    'coef_.coef_orb_weavingTRUE_StdErr': 'Orb-weaving coefficient standard error',
    'coef_.coef_orb_weavingTRUE_z.value': 'Orb-weaving coefficient z-value',
    'coef_.coef_orb_weavingTRUE_p.value': 'Orb-weaving coefficient p-value'
    }, inplace=True)

In [21]:
phyloglm_locs = convert_hogs_to_locs(phyloglm_sig, os.path.join(repo_root, "data/N5.tsv"), one_random_gene=True)
phyloglm_locs_fltrd = phyloglm_locs.drop(columns=['udiv_genes', 'dmel_orthologs', 'txpt']).drop_duplicates().reset_index(drop=True)
phyloglm_locs_fltrd = move_column(phyloglm_locs_fltrd, 'HOG', 0)
phyloglm_locs_fltrd.rename(columns={
    'LOC': 'U. diversus gene ID',
    'GO_terms': 'U. diversus GO terms',
    'Description': 'U. diversus gene description'
    }, inplace=True)
phyloglm_locs_fltrd

Processing HOGs: 100%|██████████| 2507/2507 [00:00<00:00, 98934.18it/s]


,HOG,Orb-weaving coefficient estimate,coef_orb_weavingTRUE_StdErr,coef_orb_weavingTRUE_z.value,coef_orb_weavingTRUE_p.value,qvalue,U. diversus gene ID,U. diversus GO terms,U. diversus gene description
0,N5.HOG0000608,0.273735,0.071194,3.844943,0.000121,0.001440,129232144,GO:0000981 [Name: RNA polymerase II transcript...,zinc finger protein 596-like
1,N5.HOG0001051,-0.355112,0.141780,-2.504665,0.012257,0.043956,NaN,NaN,NaN
2,N5.HOG0001055,-0.856296,0.318821,-2.685826,0.007235,0.030491,129220793,GO:0000981 [Name: RNA polymerase II transcript...,zinc finger protein 484-like
3,N5.HOG0002047,1.126103,0.334118,3.370377,0.000751,0.005910,129219201,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like
4,N5.HOG0002053,0.800547,0.178541,4.483840,0.000007,0.000163,129227284,GO:0008010 [Name: structural constituent of ch...,adult-specific rigid cuticular protein 15.7-like
...,...,...,...,...,...,...,...,...,...
2502,N5.HOG0073277,0.660198,0.237229,2.782959,0.005387,0.024678,129219304,GO:0016020 [Name: membrane]; GO:0042246 [Name:...,ninjurin-B-like
2503,N5.HOG0073283,0.662456,0.199776,3.315995,0.000913,0.006776,129225051,-,uncharacterized LOC129225051
2504,N5.HOG0073483,0.636325,0.252186,2.523239,0.011628,0.042146,129221135,GO:0043087 [Name: regulation of GTPase activity],neurofibromin-like
2505,N5.HOG0073709,-0.572810,0.218613,-2.620203,0.008788,0.034901,129229743,GO:0070034 [Name: telomerase RNA binding]; GO:...,telomerase-binding protein EST1A-like


In [22]:
phyloglm_df = get_ptep_description(phyloglm_locs_fltrd)
phyloglm_df.to_excel(os.path.join(repo_root, 'results/Supplementary_Table_7_PhyloGLM_hits.xlsx'), index=False)
phyloglm_df

,HOG,Orb-weaving coefficient estimate,coef_orb_weavingTRUE_StdErr,coef_orb_weavingTRUE_z.value,coef_orb_weavingTRUE_p.value,qvalue,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
0,N5.HOG0000608,0.273735,0.071194,3.844943,0.000121,0.001440,129232144,GO:0000981 [Name: RNA polymerase II transcript...,zinc finger protein 596-like,zinc finger protein 888
1,N5.HOG0001051,-0.355112,0.141780,-2.504665,0.012257,0.043956,NaN,NaN,NaN,oocyte zinc finger protein XlCOF15-like isofor...
2,N5.HOG0001055,-0.856296,0.318821,-2.685826,0.007235,0.030491,129220793,GO:0000981 [Name: RNA polymerase II transcript...,zinc finger protein 484-like,zinc finger protein 271-like
3,N5.HOG0002047,1.126103,0.334118,3.370377,0.000751,0.005910,129219201,GO:0008010 [Name: structural constituent of ch...,cuticle protein 10.9-like,cuticle protein 10.9
4,N5.HOG0002053,0.800547,0.178541,4.483840,0.000007,0.000163,129227284,GO:0008010 [Name: structural constituent of ch...,adult-specific rigid cuticular protein 15.7-like,adult-specific rigid cuticular protein 15.5
...,...,...,...,...,...,...,...,...,...,...
2502,N5.HOG0073277,0.660198,0.237229,2.782959,0.005387,0.024678,129219304,GO:0016020 [Name: membrane]; GO:0042246 [Name:...,ninjurin-B-like,ninjurin-2-like
2503,N5.HOG0073283,0.662456,0.199776,3.315995,0.000913,0.006776,129225051,-,uncharacterized LOC129225051,uncharacterized protein
2504,N5.HOG0073483,0.636325,0.252186,2.523239,0.011628,0.042146,129221135,GO:0043087 [Name: regulation of GTPase activity],neurofibromin-like,neurofibromin isoform X6
2505,N5.HOG0073709,-0.572810,0.218613,-2.620203,0.008788,0.034901,129229743,GO:0070034 [Name: telomerase RNA binding]; GO:...,telomerase-binding protein EST1A-like,LOW QUALITY PROTEIN: telomerase-binding protei...


In [23]:
previous_interesting_pglm_df = phyloglm_df[phyloglm_df['HOG'].isin(previous_interesting['HOG'])]
previous_interesting_pglm_df


,HOG,Orb-weaving coefficient estimate,coef_orb_weavingTRUE_StdErr,coef_orb_weavingTRUE_z.value,coef_orb_weavingTRUE_p.value,qvalue,U. diversus gene ID,U. diversus GO terms,U. diversus gene description,P. tepidariorum best BLAST hit for the HOG
39,N5.HOG0007063,0.928897,0.292805,3.172408,0.001512,0.009702,129226340,GO:0005524 [Name: ATP binding]; GO:0035556 [Na...,receptor-type guanylate cyclase Gyc76C-like,receptor-type guanylate cyclase Gyc76C
45,N5.HOG0007453,-0.886271,0.254687,-3.479845,0.000502,0.004370,129216356,GO:0043005 [Name: neuron projection],lachesin-like,lachesin
122,N5.HOG0014431,-0.816211,0.290963,-2.805209,0.005028,0.023506,129228713,GO:0016020 [Name: membrane]; GO:0007186 [Name:...,G-protein coupled receptor Mth2-like,G-protein coupled receptor Mth2-like
349,N5.HOG0026939,-0.435990,0.136184,-3.201481,0.001367,0.009043,129223346,GO:0019001 [Name: guanyl nucleotide binding]; ...,guanine nucleotide-binding protein G(q) subuni...,guanine nucleotide-binding protein G(q) subuni...
353,N5.HOG0027166,0.731890,0.238427,3.069666,0.002143,0.012388,129218261,GO:0004672 [Name: protein kinase activity]; GO...,serine/threonine-protein kinase Nek4-like,uncharacterized protein isoform X2
...,...,...,...,...,...,...,...,...,...,...
1683,N5.HOG0063259,-0.553414,0.127124,-4.353331,0.000013,0.000265,129225143,GO:0000506 [Name: glycosylphosphatidylinositol...,Phosphatidylinositol glycan anchor biosynthesi...,phosphatidylinositol N-acetylglucosaminyltrans...
1805,N5.HOG0064389,0.472255,0.097118,4.862688,0.000001,0.000039,129233173,GO:0008254 [Name: 3'-nucleotidase activity]; G...,inositol monophosphatase 3-like,inositol monophosphatase 3
2177,N5.HOG0067680,-0.363030,0.080939,-4.485232,0.000007,0.000163,129230974,GO:0004930 [Name: G-protein coupled receptor a...,transmembrane protein adipocyte-associated 1-like,transmembrane protein adipocyte-associated 1 i...
2196,N5.HOG0067795,-0.326958,0.103626,-3.155163,0.001604,0.010089,129216741,GO:0006303 [Name: double-strand break repair v...,uncharacterized LOC129216741,DNA repair protein xrcc4
